# Synthetic CBL/VDL YAML Walkthrough

This generated notebook is the recipe companion for
`examples/cbl_vdl_array_mvp_demo.py`.

Pair a synthetic in-memory dataset with a log YAML file and render it through the logfile document builder instead of the higher-level data-loading pipeline.

What you will practice in this walkthrough:

- See how a YAML-defined layout can be driven by a synthetic dataset instead of a file-backed one.
- Understand the bridge from `load_logfile(...)` to `build_documents_for_logfile(...)`.
- Render a YAML-defined array track without depending on external DLIS files.

Prerequisites:

- `pip install "wellplot[notebook]"`
- run the notebook from a checkout of this repository so the `examples/` files and sample data are available

Runtime model:

- import `wellplot` from the active installed environment
- use the repository checkout for the example files, helper modules, and sample data


In [ ]:
import sys
from pathlib import Path

try:
    import wellplot
except ImportError as exc:
    raise RuntimeError(
        "Install the published 'wellplot' package in the active "
        "environment before running this notebook."
    ) from exc

# Walk upward from the current working directory until we find the
# repository checkout that holds the example sources and sample data.
cwd = Path.cwd().resolve()
REPO_ROOT = next((path for path in (cwd, *cwd.parents) if (path / "examples").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError(
        "Run this notebook from a checkout of the wellplot repository "
        "so the example files and sample data are available."
    )

EXAMPLES_DIR = REPO_ROOT / "examples"
WORKSPACE_DIR = REPO_ROOT / "workspace"
WORKSPACE_RENDERS = WORKSPACE_DIR / "renders"
WORKSPACE_RENDERS.mkdir(parents=True, exist_ok=True)

examples_path = str(EXAMPLES_DIR)
if examples_path not in sys.path:
    sys.path.insert(0, examples_path)

print("wellplot version:", wellplot.__version__)
print("Examples root:", EXAMPLES_DIR)
print("Render output:", WORKSPACE_RENDERS)

In [ ]:
# Display the source directly in the notebook so the recipe is easy to
# read and copy from without opening another file.
from IPython.display import Code, display

source_path = EXAMPLES_DIR / "cbl_vdl_array_mvp_demo.py"
display(Code(source_path.read_text(), language="python"))

In [ ]:
# Import the legacy YAML demo helpers and load the corresponding log-file
# specification.
import cbl_vdl_array_mvp_demo as demo
from wellplot.logfile import build_documents_for_logfile, load_logfile
from wellplot.renderers import MatplotlibRenderer

logfile_path = EXAMPLES_DIR / "cbl_vdl_array_mvp.log.yaml"
spec = load_logfile(logfile_path)
dataset = demo.build_synthetic_dataset()
documents = build_documents_for_logfile(
    spec,
    dataset,
    source_path=Path("synthetic_cbl_vdl.dlis"),
)
print("Document count:", len(documents))

In [ ]:
# Render the YAML-defined documents using the same backend configuration read
# from the log-file specification.
renderer_kwargs = {"dpi": spec.render_dpi}
if spec.render_continuous_strip_page_height_mm is not None:
    renderer_kwargs["continuous_strip_page_height_mm"] = spec.render_continuous_strip_page_height_mm
style = spec.render_matplotlib.get("style")
if style is not None:
    renderer_kwargs["style"] = style

renderer = MatplotlibRenderer(**renderer_kwargs)
output_path = demo.resolve_output(logfile_path, spec.render_output_path)
result = renderer.render_documents(documents, dataset, output_path=output_path)
print("Pages:", result.page_count)
print("Rendered:", result.output_path)

## Adapt Synthetic CBL/VDL YAML Walkthrough Safely

- Use this bridge when you want a stable YAML layout but your data source is synthetic or preprocessed in memory.
- Keep the synthetic `source_path` explicit so downstream metadata stays understandable.
- Once the synthetic dataset stabilizes, switch to the higher-level pipeline only if you also want file-backed loading.
